In [1]:
import sys
import time
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
# Permite importar el paquete de src/ sin instalarlo
sys.path.insert(0, str(Path.cwd().resolve().parent / "src"))
inicio = time.perf_counter()

# 07 · Validación e interpretación · abrir la prueba, una sola vez

> Especificación: `especificaciones/07-validacion-registro-app.md` (aprobada por el grupo el 2026-11-05) · Pieza:
> `specs/003-validacion-registro-app/` · Guía: `material-del-curso/kit/guia-validacion-registro-y-app.md`, secciones 1 a 4.

Hasta la sección 4 **nadie ha mirado el conjunto de prueba**. Las reglas:

1. Las decisiones (modelo, parámetros, corrección del logaritmo, forma del precio, meta) se congelaron **antes**, el
   2026-11-05 a las 17:42, y se escriben en `reports/07_decisiones.json` antes de abrir la prueba.
2. La prueba se abre **una vez** con `open_test`, que deja la marca `reports/prueba_abierta.txt`.
3. No basta un número: intervalo por **anfitrión**, comparación con la **línea base**, la **meta**, el error por
   **ciudad** y por **tipo de habitación**, y **por qué** decide así el modelo.

Este notebook solo orquesta: la lógica está en `src/tarifalatam/models/validation.py` e `interpret.py`, con pruebas.

## 0. Datos de entrenamiento

In [2]:
import json

import joblib
import numpy as np
import pandas as pd

from tarifalatam.config import (
    FROZEN_AT,
    GROUP_COL,
    PROCESSED_DIR,
    RANDOM_STATE,
    REPORTS_DIR,
    ROOT,
    SHAP_SAMPLE,
    STAGE7_GROUP_NOTE,
    STAGE7_MODEL,
    TARGET,
    TEST_MARK_PATH,
)
from tarifalatam.data.load import load_train

ID_COL = GROUP_COL  # la entidad que se repite: el anfitrión
train = load_train()
print(f"entrenamiento: {len(train):,} anuncios de {train[ID_COL].nunique():,} anfitriones · objetivo: {TARGET}")
print("¿La prueba ya se abrió antes?", TEST_MARK_PATH.exists())

entrenamiento: 118,513 anuncios de 53,385 anfitriones · objetivo: log_price_usd
¿La prueba ya se abrió antes? False


## 1. Decisiones congeladas (antes de abrir la prueba)

Son las de la sección 1 de la especificación de la etapa. El modelo y los parámetros salen de `reports/06_best_params.json`.
La elección de LightGBM fue una **excepción a la regla** de la etapa 6, decidida después de ver su resultado (T047).

In [3]:
best = json.loads((REPORTS_DIR / "06_best_params.json").read_text(encoding="utf-8"))
assert best["finalista"] == STAGE7_MODEL, "06_best_params.json no corresponde al modelo de la etapa 7"

DECISIONES = {
    "modelo_final": STAGE7_MODEL,
    "parametros": best["parametros"],
    "parametros_fijos": best["parametros_fijos"],
    "origen_parametros": best["origen_parametros"],
    "excepcion_t047": STAGE7_GROUP_NOTE,
    "correccion_log": None,  # se llena en la sección 2, con los anfitriones reservados
    "rango_tipico": None,  # ídem
    "metrica_principal": "MAE en USD por noche",
    "metricas_de_control": ["MAPE", "R² (logaritmo)", "RMSE", "sesgo", "sesgo por ciudad", "costo 2 a 1 (1,5 y 3 a 1)"],
    "forma_del_precio": "precio puntual redondeado al dólar + rango típico de su ciudad (p25–p75 de real / sugerido en los reservados)",
    "meta": ("(a) MAE al menos 20 % menor que la línea base; (b) intervalo del 95 % de la diferencia entero bajo cero; "
             "(c) ninguna ciudad con MAE mayor que el de la línea base"),
    "congeladas_el": FROZEN_AT,
}
for clave, valor in DECISIONES.items():
    print(f"{clave}: {valor}")

modelo_final: lightgbm
parametros: {'n_estimators': 522, 'learning_rate': 0.016724373591399267, 'num_leaves': 215, 'min_child_samples': 19, 'subsample': 0.9281081091680619, 'colsample_bytree': 0.542468127069249, 'reg_lambda': 2.8855004473017765}
parametros_fijos: {'subsample_freq': 1, 'random_state': 42, 'n_jobs': -1, 'verbose': -1}
origen_parametros: optimizados
excepcion_t047: Excepción a la regla escrita, decidida por el grupo el 2026-10-31 después de ver el resultado (T047): los dos finalistas optimizados empatan en los anfitriones reservados y la regla elegiría al más rápido de ajustar; el grupo eligió LightGBM porque tiene el MAE más bajo en los reservados (dentro del ruido) y lo considera más simple de operar en la app, que carga models/model.joblib desde el repositorio.
correccion_log: None
rango_tipico: None
metrica_principal: MAE en USD por noche
metricas_de_control: ['MAPE', 'R² (logaritmo)', 'RMSE', 'sesgo', 'sesgo por ciudad', 'costo 2 a 1 (1,5 y 3 a 1)']
forma_del_precio:

## 2. Corrección del logaritmo y rango típico · con los anfitriones **reservados**, nunca con la prueba

Los mismos 25 % de anfitriones reservados de la etapa 6. El factor de Duan sale de residuos fuera de pliegue en la
búsqueda; **regla aprobada:** se usa solo si todo el intervalo de |error con factor| − |error sin factor| queda bajo cero.
Con las mismas predicciones se mide el rango típico de cada ciudad (percentiles 25 y 75 de precio real / sugerido).

In [4]:
from tarifalatam.data.folds import reserve_hosts
from tarifalatam.models.train import build_pipeline, feature_frame
from tarifalatam.models.tuning import make_model
from tarifalatam.models.validation import check_log_correction, price_range_ratios, to_usd_price

pipeline = build_pipeline(make_model(STAGE7_MODEL, DECISIONES["parametros"]), "boosting")
search, reserved = reserve_hosts(train)
correccion = check_log_correction(pipeline, search, reserved)
pred_reserved_log = correccion.pop("pred_reserved_log")
factor = correccion["factor"] if correccion["se_usa"] else 1.0

print(f"Reservados: {correccion['anuncios_reservados']:,} anuncios de {correccion['anfitriones_reservados']:,} anfitriones")
print(f"MAE sin factor: {correccion['mae_sin']:.2f} USD (en la etapa 6 dio {best['comprobacion']['mae_optimizado']:.2f})")
print(f"Factor de Duan: {correccion['factor']:.4f} · MAE con factor: {correccion['mae_con']:.2f} USD")
print(f"Diferencia (con − sin): {correccion['media']:+.2f} USD, intervalo 95 % [{correccion['low']:+.2f}; {correccion['high']:+.2f}]")
print("Decisión:", "se usa el factor" if correccion["se_usa"] else "sin corrección (el intervalo no queda entero bajo cero)")

rango = price_range_ratios(reserved["city"], reserved["price_usd"], to_usd_price(pred_reserved_log, factor))
DECISIONES["correccion_log"] = {"evaluada": True, **correccion, "factor_aplicado": factor}
DECISIONES["rango_tipico"] = rango
print(pd.DataFrame(rango, index=["p25", "p75"]).T.round(3).to_string())

Reservados: 29,023 anuncios de 13,347 anfitriones
MAE sin factor: 38.52 USD (en la etapa 6 dio 38.52)
Factor de Duan: 1.1243 · MAE con factor: 41.16 USD
Diferencia (con − sin): +2.64 USD, intervalo 95 % [+2.21; +3.06]
Decisión: sin corrección (el intervalo no queda entero bajo cero)
                  p25    p75
belize          0.744  1.354
bogota          0.781  1.149
mexico-city     0.773  1.250
rio-de-janeiro  0.724  1.277
santiago        0.815  1.152
sao-paulo       0.822  1.209


In [5]:
# Constancia de las decisiones ANTES de abrir la prueba (la hora del archivo queda antes de la marca de apertura)
DECISIONES["escritas_el"] = time.strftime("%Y-%m-%d %H:%M:%S")
(REPORTS_DIR / "07_decisiones.json").write_text(
    json.dumps(DECISIONES, indent=2, ensure_ascii=False, allow_nan=False), encoding="utf-8"
)
print("guardado: reports/07_decisiones.json ·", DECISIONES["escritas_el"])

guardado: reports/07_decisiones.json · 2026-11-05 18:09:30


## 3. Modelo final y línea base con **todo** el entrenamiento

El modelo se guarda entero (preprocesamiento + LightGBM) en `models/final_pipeline.joblib` (no se sube a Git) para
registrarlo en el notebook 08. La línea base es la mediana por ciudad × tipo de habitación, ajustada igual.

In [6]:
from sklearn.base import clone

from tarifalatam.models.train import BaselineRegressor

X_train, y_train = feature_frame(train), train[TARGET].to_numpy()
modelo_final = clone(pipeline).fit(X_train, y_train)
linea_base = build_pipeline(BaselineRegressor(), "baseline").fit(X_train, y_train)
joblib.dump(modelo_final, ROOT / "models" / "final_pipeline.joblib")
print("modelo final ajustado y guardado en models/final_pipeline.joblib")

modelo final ajustado y guardado en models/final_pipeline.joblib


## 4. Abrir la prueba · **una sola vez**

A partir de aquí **no se cambia nada del modelo**. `open_test` deja la marca con la fecha, la hora y el propósito.

In [7]:
from tarifalatam.models.validation import open_test

test = open_test(PROCESSED_DIR / "test.csv.gz", purpose=f"evaluación única de la etapa 7 (decisiones congeladas el {FROZEN_AT})")
print(TEST_MARK_PATH.read_text(encoding="utf-8").strip())
print(f"prueba: {len(test):,} anuncios de {test[ID_COL].nunique():,} anfitriones")
print(test["city"].value_counts().to_string())

Primera apertura: 2026-11-05 18:09:33 · propósito: evaluación única de la etapa 7 (decisiones congeladas el 2026-11-05 17:42 (UTC−5))
prueba: 29,721 anuncios de 13,345 anfitriones
city
sao-paulo         8283
rio-de-janeiro    8246
mexico-city       5388
bogota            3552
santiago          3399
belize             853


## 5. Métricas, intervalos por anfitrión y meta

Intervalo del 95 % remuestreando **anfitriones completos** (1.000 remuestreos, semilla 42). La meta se evalúa con la regla
escrita antes (`success_goal`), se cumpla o no.

In [8]:
from tarifalatam.models.validation import mae_with_ci, paired_mae_difference, segment_errors, success_goal, test_metrics

X_test, y_test, hosts = feature_frame(test), test[TARGET].to_numpy(), test[ID_COL]
pred_model, pred_base = modelo_final.predict(X_test), linea_base.predict(X_test)

met_model, met_base = test_metrics(y_test, pred_model, factor), test_metrics(y_test, pred_base)
ci_model, ci_base = mae_with_ci(y_test, pred_model, hosts, factor), mae_with_ci(y_test, pred_base, hosts)
diff = paired_mae_difference(y_test, pred_model, pred_base, hosts, factor_a=factor)

frame = pd.DataFrame({
    "city": test["city"], "room_type": test["room_type"], "price_usd": test["price_usd"],
    "final": to_usd_price(pred_model, factor), "linea_base": to_usd_price(pred_base),
})
segmentos = []
for by in ("city", "room_type"):
    for nombre in ("final", "linea_base"):
        tabla = segment_errors(frame.rename(columns={nombre: "pred_usd"}), by)
        segmentos.append(tabla.rename(columns={by: "valor"}).assign(segmento=by, modelo=nombre))
segmentos = pd.concat(segmentos)[["segmento", "valor", "modelo", "n", "mae", "mape", "bias", "pct_over", "n_error_over_half"]]

por_ciudad = segmentos.query("segmento == 'city'").pivot(index="valor", columns="modelo", values="mae")
city_mae = pd.DataFrame({"city": por_ciudad.index, "mae_model": por_ciudad["final"].to_numpy(),
                         "mae_baseline": por_ciudad["linea_base"].to_numpy()})
meta = success_goal(ci_model["mae"], ci_base["mae"], diff["high"], city_mae)

resumen = pd.DataFrame({"modelo final": met_model, "línea base": met_base}).T
print(resumen.round(2).to_string())
print(f"\nMAE modelo: {ci_model['mae']:.2f} USD [{ci_model['low']:.2f}; {ci_model['high']:.2f}]")
print(f"MAE línea base: {ci_base['mae']:.2f} USD [{ci_base['low']:.2f}; {ci_base['high']:.2f}]")
print(f"Diferencia (modelo − línea base): {diff['mean']:+.2f} USD [{diff['low']:+.2f}; {diff['high']:+.2f}]")
print(f"\nMeta · mejora {meta['mejora_pct']:.1f} % · (a) ≥ 20 %: {meta['a_20pct']} · (b) intervalo bajo cero: "
      f"{meta['b_intervalo']} · (c) ninguna ciudad peor: {meta['c_ciudades']} {meta['ciudades_peores'] or ''}")
print("CUMPLE LA META" if meta["cumple"] else "NO CUMPLE LA META")

                mae   mape    rmse   bias  r2_log  cost_2_1  cost_1.5_1  cost_3_1
modelo final  41.40  31.78  120.53 -14.36    0.67     54.92       48.16     68.44
línea base    57.44  42.39  159.57 -34.86    0.33     68.73       63.08     80.02

MAE modelo: 41.40 USD [38.29; 45.01]
MAE línea base: 57.44 USD [53.18; 62.50]
Diferencia (modelo − línea base): -16.04 USD [-17.84; -14.36]

Meta · mejora 27.9 % · (a) ≥ 20 %: True · (b) intervalo bajo cero: True · (c) ninguna ciudad peor: True 
CUMPLE LA META


### Error por ciudad y por tipo de habitación

In [9]:
pd.set_option("display.width", 160)
print(segmentos.round(2).to_string(index=False))

 segmento           valor     modelo     n    mae  mape    bias  pct_over  n_error_over_half
     city          belize      final   853 245.96 37.58 -109.70     45.49                226
     city          bogota      final  3552  22.32 31.64   -4.22     62.56                618
     city     mexico-city      final  5388  45.38 34.41  -17.24     52.02               1140
     city  rio-de-janeiro      final  8246  53.25 37.89  -18.79     56.55               2151
     city        santiago      final  3399  23.80 25.57   -6.64     50.81                388
     city       sao-paulo      final  8283  21.35 25.99   -5.78     54.63               1026
     city          belize linea_base   853 350.88 53.34 -266.68     36.34                397
     city          bogota linea_base  3552  27.67 37.79  -12.90     52.65                929
     city     mexico-city linea_base  5388  59.88 44.62  -34.75     48.90               1831
     city  rio-de-janeiro linea_base  8246  74.10 51.76  -45.37     49

## 6. Sensibilidad sin el filtro de precios extremos (una vez, sin decidir con ella)

La prueba de arriba excluye los precios fuera de [0,1 ×; 20 ×] la mediana de su ciudad. La app no filtra nada: aquí se
mide con **todos** los anuncios de los mismos anfitriones de prueba (se reconstruyen desde `data/raw/`).

In [10]:
from tarifalatam.data.pipeline import build_unfiltered_test

sin_filtro = build_unfiltered_test()
assert set(test["id"]) <= set(sin_filtro["id"]), "la prueba sin filtro debe contener a la filtrada"
X_sf, real_sf = feature_frame(sin_filtro), sin_filtro["price_usd"].to_numpy()
sug_sf = to_usd_price(modelo_final.predict(X_sf), factor)
base_sf = to_usd_price(linea_base.predict(X_sf))
no_positivos = int((real_sf <= 0).sum())
sensibilidad = {
    "anuncios": int(len(sin_filtro)),
    "anuncios_extremos": int(len(sin_filtro) - len(test)),
    "mae_modelo": float(np.mean(np.abs(sug_sf - real_sf))),
    "mae_linea_base": float(np.mean(np.abs(base_sf - real_sf))),
    "precios_no_positivos": no_positivos,
}
print(sensibilidad)

{'anuncios': 29807, 'anuncios_extremos': 86, 'mae_modelo': 56.805508702589314, 'mae_linea_base': 73.93925154416777, 'precios_no_positivos': 0}


## 7. ¿Por qué decide así el modelo? (SHAP)

Importancia global sobre 2.000 anuncios del **entrenamiento** y dos anuncios de la prueba: el de error mediano y el de
mayor error. Los aportes están en logaritmo (+0,10 ≈ +10 % de precio). **Explican al modelo, no al mundo**; las variables
de ubicación (latitud, longitud, barrio, ciudad) están correlacionadas y se reparten el crédito; importancia no es causa.

In [11]:
from tarifalatam.models.interpret import global_importance, shap_values, top_contributions

muestra = X_train.sample(SHAP_SAMPLE, random_state=RANDOM_STATE)
importancia = global_importance(shap_values(modelo_final, muestra))
print(importancia.round(4).to_string(index=False))

error_abs = (frame["final"] - frame["price_usd"]).abs()
casos = {"error mediano": (error_abs - error_abs.median()).abs().idxmin(), "mayor error": error_abs.idxmax()}
aportes = shap_values(modelo_final, X_test.loc[list(casos.values())])
ejemplos = {}
for posicion, (etiqueta, idx) in enumerate(casos.items()):
    fila = test.loc[idx]
    top = top_contributions(aportes, posicion, k=5)
    ejemplos[etiqueta] = {
        "ciudad": fila["city"], "room_type": fila["room_type"], "property_type": fila["property_type"],
        "accommodates": int(fila["accommodates"]), "precio_real": float(fila["price_usd"]),
        "precio_sugerido": float(frame.loc[idx, "final"]),
        "aportes": dict(zip(top["columna"], top["aporte"].round(4))),
    }
    print(f"\n{etiqueta}: {fila['city']} · {fila['room_type']} · {int(fila['accommodates'])} huéspedes · "
          f"real {fila['price_usd']:.0f} USD · sugerido {frame.loc[idx, 'final']:.0f} USD")
    print(top.round(4).to_string(index=False))

               columna  importancia
              bedrooms       0.1188
          accommodates       0.1080
             longitude       0.1018
       bathrooms_count       0.0932
              latitude       0.0785
neighbourhood_cleansed       0.0651
                  city       0.0557
         property_type       0.0445
             room_type       0.0440
        minimum_nights       0.0420
       bathroom_shared       0.0351
        maximum_nights       0.0141
                  beds       0.0103



error mediano: sao-paulo · Entire home/apt · 2 huéspedes · real 89 USD · sugerido 73 USD
               columna  aporte
       bathrooms_count -0.0778
              bedrooms -0.0726
          accommodates -0.0680
                  city -0.0642
neighbourhood_cleansed  0.0609

mayor error: belize · Private room · 16 huéspedes · real 6000 USD · sugerido 391 USD
               columna  aporte
                  city  0.7868
          accommodates  0.5524
              latitude  0.2461
neighbourhood_cleansed  0.2155
       bathrooms_count  0.1857


## 8. Guardar los resultados

`reports/07_metricas_test.json` (decisiones + prueba), `07_segmentos.csv` y `07_shap_global.csv`. Con estas cifras se
completa `reports/07_validacion.md`.

In [12]:
def _native(valor):
    return valor.item() if hasattr(valor, "item") else str(valor)


apertura = TEST_MARK_PATH.read_text(encoding="utf-8").splitlines()[0]
resultados = {
    "decisiones": DECISIONES,
    "prueba": {
        "marca": apertura,
        "anuncios": int(len(test)),
        "anfitriones": int(test[ID_COL].nunique()),
        "factor_aplicado": factor,
        "modelo": {**met_model, "mae_ic95": [ci_model["low"], ci_model["high"]]},
        "linea_base": {**met_base, "mae_ic95": [ci_base["low"], ci_base["high"]]},
        "diferencia_mae": {"media": diff["mean"], "ic95": [diff["low"], diff["high"]]},
        "meta": meta,
        "por_ciudad": city_mae.to_dict(orient="records"),
        "sensibilidad_sin_filtro": sensibilidad,
        "ejemplos_shap": ejemplos,
    },
}
(REPORTS_DIR / "07_metricas_test.json").write_text(
    json.dumps(resultados, indent=2, ensure_ascii=False, allow_nan=False, default=_native), encoding="utf-8"
)
segmentos.to_csv(REPORTS_DIR / "07_segmentos.csv", index=False)
importancia.to_csv(REPORTS_DIR / "07_shap_global.csv", index=False)
print("guardados: 07_metricas_test.json, 07_segmentos.csv, 07_shap_global.csv")
print(f"tiempo total: {(time.perf_counter() - inicio) / 60:.1f} minutos")

guardados: 07_metricas_test.json, 07_segmentos.csv, 07_shap_global.csv
tiempo total: 1.0 minutos
